# Sentiment Analysis 

This notebook implements sentiment analysis, to assign a numeric value to the sentiment of comments. 

In [1]:
import numpy as np
import sklearn
import regex
import pandas as pd

from collections import Counter
import matplotlib.pyplot as plt

from transformers import BertTokenizerFast, pipeline

from database.comments import Comments

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks

/opt/conda/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Can use the sentiment analysis pipeline from HuggingFace

In [2]:
# Example of the output
sentiment_pipeline = pipeline("sentiment-analysis")
data = ['The bricks are in keeping with the local area', 'The building will overlook ours', 'The building has windows']
sentiment_pipeline(data)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.9992750287055969},
 {'label': 'POSITIVE', 'score': 0.9976031184196472},
 {'label': 'POSITIVE', 'score': 0.9981604218482971}]

In [3]:
cs = Comments(env='dev')
df = cs.read_all()
print('df shape:', df.shape)

Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.


df shape: (30393, 13)


In [4]:
df = df[:100]

In [5]:
nlp = NLP_Tasks()

df = nlp.split_text_by_length(df, column='cleaned_comment_text', max_length=200)

Device set to use cuda:0
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Token indices sequence length is longer than the specified maximum sequence length for this model (264 > 256). Running this sequence through the model will result in indexing errors


Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.


In [6]:
df.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon,cleaned_comment_text,lsoa_code
0,97622,Barnet,24/3239/FUL_2,24/3239/FUL,12 Park Road Barnet EN5 5SQ,Objects,2024-09-20,Continued:\nThe planning statement with the cu...,2025-04-11,NaN,NaN,continued : the planning statement with the cu...,None
1,97622,Barnet,24/3239/FUL_2,24/3239/FUL,12 Park Road Barnet EN5 5SQ,Objects,2024-09-20,Continued:\nThe planning statement with the cu...,2025-04-11,NaN,NaN,destroys any sense of community on the street ...,None
2,87894,Lambeth,20/04194/EIAFUL_123,20/04194/EIAFUL,None,Objects,2021-01-27,Lambeth have pushed all the traffic on to Land...,2025-04-10,NaN,NaN,have pushed all the traffic on to causing grid...,None
3,87897,Lambeth,20/04194/EIAFUL_126,20/04194/EIAFUL,None,Objects,2021-01-26,"Consistent with other residents, I object to t...",2025-04-10,NaN,NaN,"consistent with other residents, i object to t...",None
4,87897,Lambeth,20/04194/EIAFUL_126,20/04194/EIAFUL,None,Objects,2021-01-26,"Consistent with other residents, I object to t...",2025-04-10,NaN,NaN,"opinions, and not allow the developers to hide...",None


In [7]:
data = df['cleaned_comment_text'].to_list()
data_stance = df['stance'].to_list()

In [8]:
sentiment_pipeline(data)

[{'label': 'NEGATIVE', 'score': 0.9996985197067261},
 {'label': 'NEGATIVE', 'score': 0.9996806383132935},
 {'label': 'NEGATIVE', 'score': 0.9997561573982239},
 {'label': 'NEGATIVE', 'score': 0.9982609152793884},
 {'label': 'NEGATIVE', 'score': 0.9954083561897278},
 {'label': 'NEGATIVE', 'score': 0.9861704111099243},
 {'label': 'NEGATIVE', 'score': 0.9995379447937012},
 {'label': 'NEGATIVE', 'score': 0.9982319474220276},
 {'label': 'NEGATIVE', 'score': 0.9995813965797424},
 {'label': 'POSITIVE', 'score': 0.8650698065757751},
 {'label': 'NEGATIVE', 'score': 0.9993032217025757},
 {'label': 'NEGATIVE', 'score': 0.9992259740829468},
 {'label': 'NEGATIVE', 'score': 0.9988382458686829},
 {'label': 'POSITIVE', 'score': 0.8064026236534119},
 {'label': 'POSITIVE', 'score': 0.9998457431793213},
 {'label': 'NEGATIVE', 'score': 0.9987298846244812},
 {'label': 'NEGATIVE', 'score': 0.9991436004638672},
 {'label': 'NEGATIVE', 'score': 0.9989832043647766},
 {'label': 'NEGATIVE', 'score': 0.999696612358

In [9]:
sentiment_model = pipeline(model="finiteautomata/bertweet-base-sentiment-analysis")

comment_a = 'The bricks are in keeping with the local area. But the building will overlook ours. Which I hate.'

# Split the comment into sentences
sentences = regex.split(r'(?<=[.!?]) +', comment_a)

print('Sentences:', sentences)

# Analyze sentiment for each sentence
sentiment_results = sentiment_model(sentences)
for sentence, result in zip(sentences, sentiment_results):
    print(f"Sentence: {sentence}\nSentiment: {result['label']}, Score: {result['score']:.4f}\n")

Device set to use cuda:0


Sentences: ['The bricks are in keeping with the local area.', 'But the building will overlook ours.', 'Which I hate.']
Sentence: The bricks are in keeping with the local area.
Sentiment: NEU, Score: 0.6146

Sentence: But the building will overlook ours.
Sentiment: NEU, Score: 0.7616

Sentence: Which I hate.
Sentiment: NEG, Score: 0.9799



In [10]:
sentiment_results

[{'label': 'NEU', 'score': 0.6145685911178589},
 {'label': 'NEU', 'score': 0.7615547180175781},
 {'label': 'NEG', 'score': 0.9798897504806519}]

In [11]:
n = len(sentiment_results)

# calculate score by adding 'POS' scores and subtracting 'NEG' scores
score = 0
for result in sentiment_results:
    if result['label'] == 'POS':
        score += result['score']
    elif result['label'] == 'NEG':
        score -= result['score']

score = score / n

print(f"Overall sentiment score: {score:.4f}")

Overall sentiment score: -0.3266


In [12]:
comment_a = 'The bricks are in keeping with the local area. But the building will overlook ours. Which I hate.'

In [ ]:
import io
import contextlib

def quiet_pipeline(*args, **kwargs):
    f = io.StringIO()
    with contextlib.redirect_stdout(f):
        return pipeline(*args, **kwargs)

def sentiment_score(comment):
    sentiment_model = quiet_pipeline(model="finiteautomata/bertweet-base-sentiment-analysis", device=0)

    # Split the comment into sentences
    sentences = regex.split(r'(?<=[.!?]) +', comment)
    n = len(sentences)

    # print('Sentences:', sentences)

    # Analyse sentiment for each sentence
    sentiment_results = sentiment_model(sentences)

    # Calculate score by adding 'POS' scores and subtracting 'NEG' scores
    score = 0
    for result in sentiment_results:
        if result['label'] == 'POS':
            score += result['score']
        elif result['label'] == 'NEG':
            score -= result['score']

    score = score / n
    # print(f"Overall sentiment score: {score:.4f}")

    return float(score)

In [29]:
sentiment_score('I love this so much, but I hate traffic.')

Device set to use cuda:0


0.8191016316413879

In [30]:
conflict_comments = []
conflict_comments_score = []
for i in range(len(data)):
    comment = data[i]
    score = sentiment_score(comment)
    if score >= 0 and data_stance[i] == 'Objects':
        conflict_comments.append(data[i])
        conflict_comments_score.append(score)

Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0


In [31]:
for item in zip(conflict_comments, conflict_comments_score):
    print(f"Comment: {item[0]}\nSentiment Score: {item[1]:.4f}\n")

Comment: resident that supports the proposal in its current form, then should it be allowed to proceed without serious / sensible interrogation and modification, it would be indicative of the councils ineffectual handling of this planning application and an unwillingness to serve and protect its residents in the way that it is bound to. i was heartened to read recently that that london mayor sadiq khan has tightened the city ' s policy on tall buildings ( after an intervention by robert jenrick, the secretary of state for housing, communities and local government ) and that as such tall buildings are only brought forward in appropriate and clearly defined areas ". landor road and it surrounds, is clearly not an appropriate area. a plan more akin to the oval quarter would be more appropriate - lower rise, lower density and with a clear value exchange in the benefit that it adds to the community through green space and facilities for use by all. thank you.
Sentiment Score: 0.1061

Commen

In [32]:
len(conflict_comments)

16